# Encoding TEI with LLMs

This notebook demonstrates how to use Large Language Models (LLMs) to assist in the encoding of texts in the Text Encoding Initiative (TEI) format. The TEI is a standard for representing texts in digital form, and LLMs can help automate and enhance the encoding process.


In [7]:
from pathlib import Path
import os
from dotenv import load_dotenv
from openai import OpenAI

from tqdm import tqdm

In [8]:
# =====================================================
# PROMPT CENTRALISÉ
# =====================================================

SYSTEM_PROMPT = """
Tu es un expert en encodage XML conforme à la TEI Lex-0 pour dictionnaires anciens du XVIIIe siècle.

Tu traites des pages OCR issues du Dictionnaire de Trévoux (édition 1743).

OBJECTIF

Transformer un texte OCR brut en XML TEI Lex-0 structuré.

Tu dois :
1. Identifier automatiquement le début et la fin de chaque entrée.
2. Identifier les sous-entrées éventuelles.
3. Structurer chaque entrée selon le modèle TEI Lex-0 fourni.
4. Ne produire QUE du XML valide.
5. Ne produire aucun commentaire, aucune explication.

IMPORTANT

- Ne rien inventer.
- Ne pas corriger l’OCR.
- Ne pas moderniser l’orthographe.
- Conserver la ponctuation et la casse d’origine.
- Ne pas compléter un mot tronqué.
- Si un élément est ambigu, l’encoder sans interprétation.
- Vérifier que chaque balise ouverte est fermée.

FORMAT DE SORTIE OBLIGATOIRE

<TEI>
  <text>
    <body>
      ... entrées encodées ...
    </body>
  </text>
</TEI>

STRUCTURE AUTORISÉE (STRICTEMENT)

<entry>
  <form type="lemma">
    <orth>...</orth>
  </form>
  <gramGrp>
    <pos>...</pos>
  </gramGrp>
  <sense>
    <def>...</def>
    <cit type="example">
      <quote>...</quote>
    </cit>
  </sense>
</entry>

RÈGLES DE DÉTECTION DES ENTRÉES (TRÉVOUX 1743)

Une nouvelle entrée commence si :

- Un mot ou groupe de mots est entièrement en capitales
- Il apparaît en début de paragraphe
- Il est suivi immédiatement d’une virgule ou d’un point
- Il est suivi d’une indication grammaticale (s. m., s. f., v. a., adj., etc.)
- Il y a rupture typographique claire dans l’OCR

Le lemme correspond au premier segment en capitales avant la première virgule ou marque grammaticale.

RÈGLES POUR LES CATÉGORIES GRAMMATICALES

Si une abréviation grammaticale suit immédiatement le lemme :
- l’encoder dans <pos> sans modification
- conserver exactement la forme OCR (ex : “s. m.”, “v. act.”)

Si aucune catégorie grammaticale n’est identifiable :
- ne pas produire <gramGrp>

RÈGLES POUR LES SENS

Créer un nouveau <sense> si :

- présence de numérotation (1., 2., I., II.)
- présence de lettres (A., B.)
- rupture sémantique claire introduite par une marque visible

Sinon :
- créer un seul <sense> pour toute l’entrée

RÈGLES POUR LES DÉFINITIONS

- Tout texte définitoire doit être placé dans <def>.
- Conserver intégralement le texte OCR.
- Ne pas reformuler.

RÈGLES POUR LES CITATIONS

Encoder comme <cit type="example"> si :

- phrase en italique signalée par l’OCR
- citation introduite par nom d’auteur
- citation latine
- citation entre guillemets

Structure :

<cit type="example">
  <quote>texte exact</quote>
</cit>

Ne jamais séparer une citation en plusieurs blocs.

RENVOIS INTERNES

Si présence de “Voyez …” ou “V.” :
- laisser le texte dans <def>
- ne pas créer de balise spéciale

CAS PARTICULIERS

- Si l’entrée est incomplète (coupure de page), l’encoder telle quelle.
- Si deux entrées semblent fusionnées, les séparer uniquement si le début d’entrée est clairement détectable.
- Ne pas créer d’attributs XML supplémentaires.
- Ne pas ajouter de balises non spécifiées.

EXEMPLE

TEXTE OCR :

ABANDON, s. m. Action de délaisser. Ex. L’abandon d’un bien.

SORTIE :

<entry>
  <form type="lemma">
    <orth>ABANDON</orth>
  </form>
  <gramGrp>
    <pos>s. m.</pos>
  </gramGrp>
  <sense>
    <def>Action de délaisser.</def>
    <cit type="example">
      <quote>L’abandon d’un bien.</quote>
    </cit>
  </sense>
</entry>

PROCÉDURE INTERNE (NE PAS AFFICHER)

1. Segmenter les entrées
2. Identifier lemma
3. Identifier catégorie grammaticale
4. Segmenter les sens
5. Identifier citations
6. Générer XML
7. Vérifier fermeture des balises

"""


def build_user_prompt(text: str) -> str:
    return f"""TEXTE À ENCODER :
{text}

TEXTE ENCODE EN XML TEI LEX-0 :
"""

# =====================================================
# BACKENDS
# =====================================================

def ocr_openrouter(client, model_name, text):
    user_prompt = build_user_prompt(text)
    # Gemma → pas de system message
    if "gemma" in model_name.lower() or "mistral" in model_name.lower():

        full_prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "user", "content": full_prompt},
            ],
        )

    # Autres modèles → messages normaux
    else:
        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )

    return response.choices[0].message.content.strip()




# =====================================================
# INTERFACE UNIQUE
# =====================================================

def post_ocr_correction(text, provider, client, model_name):
    try:
        if provider == "openrouter":
            return ocr_openrouter(client, model_name, text)
        else:
            raise ValueError("Provider inconnu")

    except Exception as e:
        print(f"Erreur {provider}: {e}")
        return text


# =====================================================
# INITIALISATION
# =====================================================

load_dotenv()



True

In [ ]:
# =====================================================
# USAGE (example de test rapide)
# =====================================================

text = "Ceci est un eximp1e de texte OQR avec des emeurs."

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

model_name = "google/gemma-3-27b-it"

print("OpenRouter (gemma):", post_ocr_correction(text, "openrouter", openrouter_client, model_name))

In [9]:
'''
openrouter_client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm_studio",
)
models = ["mistralai/mistral-7b-instruct-v0.3"]
'''

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
models = ["openai/gpt-5-mini"] #, "google/gemma-3-27b-it", "google/gemma-3-12b-it"]


ocr_path_name = "data/glm-ocr/txt/"
ocr_files = sorted(Path(ocr_path_name).glob("*.txt"))

for model_name in models:
    model_short = model_name.split("/")[1].split(":")[0]

    # create folder with model name if it doesn't exist
    output_path = Path(f"data/encoded/{model_short}")
    output_path.mkdir(parents=True, exist_ok=True)

    for ocr_file in tqdm(ocr_files):
        ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

        # if file already exists, skip
        output_file = output_path / f"{ocr_file.stem}.xml"
        if output_file.exists():
            continue
        corrected_text = post_ocr_correction(ocr_text, "openrouter", openrouter_client, model_name)

        # Write corrected text to output file
        output_file.write_text(corrected_text, encoding="utf-8", errors="replace")
  



100%|██████████| 1/1 [02:27<00:00, 147.07s/it]
